# LightGBM: hydropower unit commitment

**Group number:** TODO  
**Full names and student IDs:** TODO  
**Kaggle team name:** TODO

One separate binary LightGBM classifier predicts each of the 14 generators.
Each row represents one scheduling case and one horizon hour. The default uses
the same 216 decision-context features, target ordering and purged chronological
development folds as the CatBoost decision/RSM experiments. This tests a different
tree learner on the same information. Equal feature inputs do not imply equal
model capacity or identically tuned models.

All required preprocessing and training code is included here; this notebook does
not import another notebook, source helper, saved configuration or trained model.
The user installs `requirements.txt` and runs all training. No downloads or package
installation occur during notebook execution.

| Mode | What runs |
|---|---|
| `check` (default) | Data/feature/schema checks, no training; LightGBM need not be installed |
| `smoke` | 96 training and 48 validation cases from the 2021 fold, at most 40 trees per unit |
| `validate` | Two development folds (2020 and 2021), 14 models per fold, Logloss stopping |
| `holdout` | Frozen configuration fitted before 2022, evaluated on 2022 without early stopping |
| `submit` | Frozen configuration fitted on all labels; export 725 test cases |

Restart the kernel and run all cells after changing settings. Start with `check`,
install dependencies yourself, then run `smoke` before `validate`. Runtime and
predictive quality have not been measured. Do not assume this beats CatBoost.
After validation, paste the printed LightGBM configuration into `SELECTED_CONFIG`.
CatBoost configurations and tree counts are rejected even if feature names match.
Each training run writes a new directory under `outputs/lightgbm/`.

The public Kaggle score is a development signal, not a grading guarantee. Preserve
2022 for final frozen-candidate confirmation; the historical baseline has already
been evaluated there, so it is not wholly unseen across project history.


### Controlled optimization presets

The current preset is `decision_neutral_logloss` in `check` mode. It retains the
216 features, folds, seed and model capacity, changing only `boost_from_average`
to False. For binary classification with no supplied initial score this starts
from probability 0.5, rather than the training class proportion. This targets
possible sensitivity to class-proportion drift, not a guaranteed improvement.

Later separate tests: `decision_regularized_logloss` reduces capacity (31 leaves,
depth 6, min_child_samples 500, L2 10), while `decision_slow_logloss` changes only
learning rate to 0.03. They retain the original prior initialization; they do not
combine with the neutral preset. All use the same 2000-tree ceiling and patience
100. Slower learning may require a longer patience/budget if learning curves show
that is justified; do not increase it without checking actual results. Run each
one separately with validate. Do not pick models solely on 2021 or Kaggle scores.

The first completed LightGBM configuration is preserved as
`LIGHTGBM_REFERENCE_CONFIG`, usable with `EXPERIMENT = "decision_logloss"`. Each
new preset must generate its own selected configuration. CatBoost is unchanged.


### Completed optimization batch

The assistant ran the neutral and regularized development experiments under the
user's explicit 45-minute authorization. Neutral initialization scored 0.964358
(2020) / 0.983371 (2021); regularized scored 0.959672 / 0.983777. Neutral was
selected for the stronger balanced development result. The slower-learning preset
was deferred for budget reasons. The neutral configuration is now frozen inline in
SELECTED_CONFIG; current mode remains check. The selected full-data model and a
75% CatBoost RSM / 25% neutral-LightGBM blended submission have been generated.
Neither has a measured Kaggle score. See the experiment record for paths and
reproduction requirements; no output artifact is a runtime dependency here.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from time import perf_counter
import gc
import hashlib
import json
import platform
import re

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd()
RUN_MODE = "check"  # check, smoke, validate, holdout, submit
EXPERIMENT = "decision_neutral_logloss"
MODEL_FAMILY = "lightgbm"
MODEL_STRATEGY = "per_generator"
SEED = 42
THREADS = 4
HOURS = 168
EXPERIMENTS = {
    "decision_neutral_logloss": {"features": "hydro-decision-v1",
                                 "decision_groups": ["economics", "water_balance"],
                                 "parameter_overrides": {"boost_from_average": False}},
    "decision_regularized_logloss": {"features": "hydro-decision-v1",
                                     "decision_groups": ["economics", "water_balance"],
                                     "parameter_overrides": {"num_leaves": 31, "max_depth": 6,
                                                             "min_child_samples": 500, "reg_lambda": 10.0}},
    "decision_slow_logloss": {"features": "hydro-decision-v1",
                              "decision_groups": ["economics", "water_balance"],
                              "parameter_overrides": {"learning_rate": 0.03}},
    "decision_logloss": {"features": "hydro-decision-v1",
                         "decision_groups": ["economics", "water_balance"]},
    "baseline_logloss": {"features": "hydro-context-v1", "decision_groups": []},
}
if EXPERIMENT not in EXPERIMENTS:
    raise ValueError("Unknown LightGBM experiment preset.")
FEATURE_VERSION = EXPERIMENTS[EXPERIMENT]["features"]
DECISION_GROUPS = set(EXPERIMENTS[EXPERIMENT]["decision_groups"])
FOLDS = [("2020-01-01", "2021-01-01"), ("2021-01-01", "2022-01-01")]
HOLDOUT_START = pd.Timestamp("2022-01-01", tz="UTC")
EARLY_STOPPING = 100
MAX_ITERATIONS = 2000
# Explicit starting values, not tuned values. Tune on development folds only.
BASE_PARAMETERS = {
    "objective": "binary", "boosting_type": "gbdt", "metric": "binary_logloss",
    "learning_rate": 0.05, "num_leaves": 63, "max_depth": 7,
    "min_child_samples": 200, "reg_lambda": 5.0, "reg_alpha": 0.0,
    "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8,
    "max_bin": 127, "device_type": "cpu", "deterministic": True,
    "force_col_wise": True, "verbosity": -1,
}
BASE_PARAMETERS.update(EXPERIMENTS[EXPERIMENT].get("parameter_overrides", {}))

# Keep the measured first LightGBM configuration as a reproducible reference.
LIGHTGBM_REFERENCE_CONFIG = {'model_family': 'lightgbm',
 'experiment': 'decision_logloss',
 'model_strategy': 'per_generator',
 'feature_version': 'hydro-decision-v1',
 'feature_signature': '413b34b792c90dc7fb586250679bd5438ef0dc4e90ef8b07a8bdb6331b6c2af9',
 'seed': 42,
 'parameters': {'objective': 'binary',
                'boosting_type': 'gbdt',
                'metric': 'binary_logloss',
                'learning_rate': 0.05,
                'num_leaves': 63,
                'max_depth': 7,
                'min_child_samples': 200,
                'reg_lambda': 5.0,
                'reg_alpha': 0.0,
                'subsample': 0.8,
                'subsample_freq': 1,
                'colsample_bytree': 0.8,
                'max_bin': 127,
                'device_type': 'cpu',
                'deterministic': True,
                'force_col_wise': True,
                'verbosity': -1},
 'iterations': {'Hogga_G1': 210,
                'Tokke_G1': 208,
                'Tokke_G2': 176,
                'Tokke_G3': 345,
                'Tokke_G4': 283,
                'Lio_G1': 110,
                'Byrte_G1': 443,
                'Vinje_G1': 162,
                'Vinje_G2': 165,
                'Vinje_G3': 286,
                'Haukeli_G1': 79,
                'Songa_G1': 196,
                'Kjela_G1': 358,
                'Vesle Kjela_G1': 367}}

# Frozen neutral-initialization configuration from completed assistant-run validation.
SELECTED_CONFIG = {'model_family': 'lightgbm',
 'experiment': 'decision_neutral_logloss',
 'model_strategy': 'per_generator',
 'feature_version': 'hydro-decision-v1',
 'feature_signature': '413b34b792c90dc7fb586250679bd5438ef0dc4e90ef8b07a8bdb6331b6c2af9',
 'seed': 42,
 'parameters': {'objective': 'binary',
                'boosting_type': 'gbdt',
                'metric': 'binary_logloss',
                'learning_rate': 0.05,
                'num_leaves': 63,
                'max_depth': 7,
                'min_child_samples': 200,
                'reg_lambda': 5.0,
                'reg_alpha': 0.0,
                'subsample': 0.8,
                'subsample_freq': 1,
                'colsample_bytree': 0.8,
                'max_bin': 127,
                'device_type': 'cpu',
                'deterministic': True,
                'force_col_wise': True,
                'verbosity': -1,
                'boost_from_average': False},
 'iterations': {'Hogga_G1': 257,
                'Tokke_G1': 235,
                'Tokke_G2': 184,
                'Tokke_G3': 321,
                'Tokke_G4': 256,
                'Lio_G1': 100,
                'Byrte_G1': 406,
                'Vinje_G1': 204,
                'Vinje_G2': 111,
                'Vinje_G3': 302,
                'Haukeli_G1': 69,
                'Songa_G1': 203,
                'Kjela_G1': 417,
                'Vesle Kjela_G1': 358}}

if RUN_MODE not in {"check", "smoke", "validate", "holdout", "submit"}:
    raise ValueError("Unknown RUN_MODE.")
if not (ROOT / "prediction_mapping.csv").is_file():
    raise FileNotFoundError("Run this notebook from the project root with the provided raw data present.")
if not 1 <= THREADS <= 4:
    raise ValueError("Use between 1 and 4 CPU threads for the course reproduction budget.")
NOTEBOOK_STARTED = perf_counter()
VERSIONS = {"python": platform.python_version()}
for package in ("numpy", "pandas", "PyYAML", "lightgbm", "scikit-learn", "matplotlib"):
    try:
        VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        VERSIONS[package] = "not installed"
print("Mode:", RUN_MODE, "Model:", MODEL_FAMILY, "Experiment:", EXPERIMENT, "Features:", FEATURE_VERSION)
print("Package versions:", VERSIONS)
if RUN_MODE != "check":
    missing = [p for p in ("lightgbm", "scikit-learn") if VERSIONS[p] == "not installed"]
    if missing:
        raise ImportError(f"Install requirements.txt into this kernel's environment yourself before training: {missing}")


## 1. Case definitions and target schema

`Run No` is an identifier, not a feature. `total_calculation_time` is optimizer output
and is excluded. Target columns are parsed by name rather than assumed reshape order.
The sample submission provides the exact test IDs and output ordering.

In [ ]:
RAW_FILES = {
    "labels": ROOT / "data/kernel/Unit_commitment_decisions.csv",
    "mapping": ROOT / "prediction_mapping.csv",
    "template": ROOT / "sample_submission.csv",
    "prices": ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "inflows": ROOT / "data/kernel/Historical_inflow_1958_2025.csv",
    "volumes": ROOT / "data/kernel/Historical_volume_2015_2024.csv",
    "water_values": ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv",
    "min_flow": ROOT / "data/extended/Constraint_min_flow.csv",
    "min_volume": ROOT / "data/extended/Constraint_min_volume.csv",
    "topology": ROOT / "data/extended/Tokke_Vinje_topology.yaml",
}
for path in RAW_FILES.values():
    if not path.is_file():
        raise FileNotFoundError(path)
train_cases = pd.read_csv(RAW_FILES["labels"])
test_cases = pd.read_csv(RAW_FILES["mapping"])
template = pd.read_csv(RAW_FILES["template"])
for cases in (train_cases, test_cases):
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, errors="raise")
    if cases.empty or cases["Run No"].isna().any() or not cases["Run No"].is_unique:
        raise ValueError("Cases require nonempty, unique, nonmissing Run No values.")
    if cases["starttime"].isna().any() or not cases["starttime"].is_unique:
        raise ValueError("Case start times must be unique and nonmissing.")
    if not cases["starttime"].equals(cases["starttime"].dt.normalize()):
        raise ValueError("Expected midnight UTC case starts.")
if not template["Run No"].is_unique or set(template["Run No"]) != set(test_cases["Run No"]):
    raise ValueError("Submission IDs must match the test mapping exactly.")
if set(train_cases["Run No"]) & set(test_cases["Run No"]):
    raise ValueError("Training and test case IDs overlap.")
train_cases = train_cases.sort_values("starttime").reset_index(drop=True)
test_cases = test_cases.set_index("Run No").loc[template["Run No"]].reset_index()
LABELS = template.columns.drop("Run No").tolist()
TARGETS = {}
for position, name in enumerate(LABELS):
    match = re.fullmatch(r"result_committed_(.+_G\d+)_t(\d+)", name)
    if match is None:
        raise ValueError(f"Unexpected target column: {name}")
    generator, hour = match.group(1), int(match.group(2))
    if hour in TARGETS.setdefault(generator, {}):
        raise ValueError(f"Duplicate generator/hour: {generator}, {hour}")
    TARGETS[generator][hour] = (name, position)
GENERATORS = list(TARGETS)
if len(GENERATORS) != 14 or len(LABELS) != 14 * HOURS:
    raise ValueError("Expected 14 generators and 2352 targets.")
for generator in GENERATORS:
    if set(TARGETS[generator]) != set(range(HOURS)):
        raise ValueError(f"Incomplete horizon for {generator}")
TARGET_COLUMNS = {g: [TARGETS[g][h][0] for h in range(HOURS)] for g in GENERATORS}
TARGET_POSITIONS = {g: [TARGETS[g][h][1] for h in range(HOURS)] for g in GENERATORS}
if not set(LABELS).issubset(train_cases) or not train_cases[LABELS].isin([0, 1]).all().all():
    raise ValueError("All target columns must be present with binary, nonmissing labels.")

def labels_for(cases, generator):
    return cases[TARGET_COLUMNS[generator]].to_numpy(dtype=np.uint8).reshape(-1)

all_starts = pd.DatetimeIndex(pd.concat([train_cases["starttime"], test_cases["starttime"]]))
FIRST_START = all_starts.min()
LAST_END = all_starts.max() + pd.Timedelta(hours=HOURS)
print(f"{len(train_cases)} training cases; {len(test_cases)} test cases; {len(LABELS)} targets")
display(pd.DataFrame({"generator": GENERATORS,
    "training_on_fraction": [float(labels_for(train_cases, g).mean()) for g in GENERATORS]}))

## 2. Raw inputs, units, and topology

Prices and inflows cover the planning horizon; volumes are taken **only at the case
start**, and terminal water values **only at start + 168 hours**. Environmental limits
are taken within the horizon. Every lookup is timestamp-based in UTC. No interpolation
or silent zero-filling is used for missing records. The later quarter-hour price
records outside the task's required date range are not resampled into our data.

We use the provided YAML reservoir capacities to derive filling fractions, and its
connections to identify each plant's immediate supplying reservoirs, traversing
intake tunnels. These local aggregates are descriptive features, not a hydraulic
simulator or proof of physical feasibility. All reservoirs also remain available
individually to every generator model.

In [ ]:
def read_timeseries(key):
    frame = pd.read_csv(RAW_FILES[key])
    dates = pd.to_datetime(frame.pop("date"), utc=True, errors="raise")
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f"Invalid or duplicate timestamps in {key}")
    frame.index = pd.DatetimeIndex(dates, name="date")
    frame = frame.sort_index().loc[FIRST_START:LAST_END]
    frame = frame.apply(pd.to_numeric, errors="raise").astype(np.float32)
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Empty or nonfinite input: {key}")
    return frame

tables = {key: read_timeseries(key) for key in
          ("prices", "inflows", "volumes", "water_values", "min_flow", "min_volume")}
if list(tables["prices"].columns) != ["Day-ahead price (EUR/MWh)"]:
    raise ValueError("Unexpected electricity price column.")
with RAW_FILES["topology"].open(encoding="utf-8") as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = list(tables["volumes"].columns)
if len(RESERVOIRS) != 17:
    raise ValueError("Expected 17 reservoirs.")
for key in ("inflows", "water_values"):
    if not set(RESERVOIRS).issubset(tables[key].columns):
        raise ValueError(f"Missing reservoir columns in {key}")
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}
if any(v <= 0 for v in CAPACITY.values()):
    raise ValueError("Reservoir capacities must be positive.")
for key in ("inflows", "volumes", "min_flow", "min_volume"):
    if (tables[key].to_numpy() < 0).any():
        raise ValueError(f"Unexpected negative water quantity in {key}")

hourly_required = pd.date_range(FIRST_START, LAST_END, freq="h", inclusive="left")
for key in ("prices", "inflows", "min_flow", "min_volume"):
    used = tables[key].loc[tables[key].index < LAST_END]
    if not used.index.equals(hourly_required):
        raise ValueError(f"{key} must contain exactly one record per required hour.")
for key, timestamps in (("volumes", all_starts),
                        ("water_values", all_starts + pd.Timedelta(hours=HOURS))):
    if not timestamps.difference(tables[key].index).empty:
        raise ValueError(f"Missing required daily records in {key}")

connections = topology["connections"]
def supplying_reservoirs(name, kind, visited=None):
    visited = set() if visited is None else visited
    node = (kind, name)
    if node in visited:
        raise ValueError("Cycle in intake topology.")
    if kind == "reservoir":
        return {name}
    result = set()
    for edge in connections:
        if edge["to"] == name and edge["to_type"] == kind and edge["from_type"] in {"reservoir", "tunnel"}:
            result |= supplying_reservoirs(edge["from"], edge["from_type"], visited | {node})
    return result

FEEDERS = {}
for generator in GENERATORS:
    plants = [e["to"] for e in connections if e["from"] == generator
              and e["from_type"] == "generator" and e["to_type"] == "plant"]
    if len(plants) != 1:
        raise ValueError(f"Expected one plant for {generator}")
    FEEDERS[generator] = sorted(supplying_reservoirs(plants[0], "plant"))
    if not FEEDERS[generator] or not set(FEEDERS[generator]).issubset(RESERVOIRS):
        raise ValueError(f"Unknown supplying reservoirs for {generator}")
display(pd.DataFrame({"generator": GENERATORS,
                      "intake_reservoirs": [", ".join(FEEDERS[g]) for g in GENERATORS]}))
display(pd.DataFrame([{"input": k, "rows_in_task_range": len(v), "columns": len(v.columns),
                       "start": str(v.index.min()), "end": str(v.index.max())}
                      for k, v in tables.items()]))

# Traverse every supplied hydraulic predecessor, including river reaches and plants.
# A visited set prevents repeated counting at merges or cycles. These are possible
# upstream connections, not evidence that all upstream water arrives in this horizon.
def upstream_reservoirs(generator):
    plant = next(e["to"] for e in connections if e["from"] == generator
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    pending, visited, reservoirs = [("plant", plant)], set(), set()
    while pending:
        kind, name = pending.pop()
        if (kind, name) in visited:
            continue
        visited.add((kind, name))
        if kind == "reservoir":
            reservoirs.add(name)
        pending.extend((edge["from_type"], edge["from"]) for edge in connections
                       if edge["to_type"] == kind and edge["to"] == name
                       and edge["from_type"] in {"reservoir", "river", "tunnel", "plant"})
    return sorted(reservoirs)

UPSTREAM = {g: upstream_reservoirs(g) for g in GENERATORS}
OUTLET_ELEVATION = {}
for g in GENERATORS:
    plant = next(e["to"] for e in connections if e["from"] == g
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    OUTLET_ELEVATION[g] = float(topology["model"]["plant"][plant]["outlet_line"])
    if not set(FEEDERS[g]).issubset(UPSTREAM[g]) or not set(UPSTREAM[g]).issubset(RESERVOIRS):
        raise ValueError(f"Invalid upstream topology for {g}")
if FEATURE_VERSION == "hydro-context-v2":
    display(pd.DataFrame({"generator": GENERATORS,
                          "all_upstream_reservoirs": [", ".join(UPSTREAM[g]) for g in GENERATORS]}))


## 3. Feature engineering

Feature construction below is copied exactly from the verified CatBoost pipeline.
The default `decision_logloss` has 216 numeric features: original 193 operating and
weekly-context features plus 14 economic-period and 9 water-pressure descriptors.
`baseline_logloss` provides the 193-feature input ablation.

Use price/inflow/constraints inside [S, S+168h), volume at S only, and water value
at S+168h. Joins use UTC timestamps and named objects. Neither optimizer runtime,
held-out commitment labels nor observed future volumes are model features.

Economic descriptors use price minus capacity-weighted terminal water value for
the plant's intake reservoirs, positive-margin streaks/distances, remaining
opportunities and truncated 6h/24h windows. They are not true net-profit estimates.
Water-pressure descriptors integrate natural inflow to Mm3 (3600/1e6), compare
against initial headroom, and calculate hypothetical excess/floor margins assuming
zero release or transfer. Per-reservoir excesses are computed before aggregation.
The headroom ratio denominator has a 1%-capacity floor and uses log1p. These are
descriptors, not a simulator or forecasts of realized storage. Windows include the
current hour and stay inside each case. Distance 168 means no positive-margin
hour exists in the relevant direction within the horizon.

No learned preprocessing is fitted before a chronological split. LightGBM fits
numeric binning on training inputs; validation data is used for development
stopping only. The reserved holdout is not used for stopping or model selection.


In [ ]:
def economic_context(margin, price):
    """Whole-horizon price/value descriptors, not a dispatch profitability solver."""
    margin, price = np.asarray(margin, dtype=np.float64), np.asarray(price, dtype=np.float64)
    n, width = margin.shape
    if width != HOURS or price.shape != margin.shape:
        raise ValueError("Economic context requires one complete horizon per row.")
    positive = margin > 0
    positive_margin = np.maximum(margin, 0)
    left_length = np.zeros_like(margin)
    right_length = np.zeros_like(margin)
    left_sum = np.zeros_like(margin)
    right_sum = np.zeros_like(margin)
    next_positive = np.full_like(margin, HOURS)
    previous_positive = np.full_like(margin, HOURS)
    # Each recurrence resets independently for every scheduling case.
    for h in range(width):
        left_length[:, h] = np.where(positive[:, h], (left_length[:, h-1] if h else 0) + 1, 0)
        left_sum[:, h] = np.where(positive[:, h], (left_sum[:, h-1] if h else 0) + margin[:, h], 0)
        previous_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((previous_positive[:, h-1] if h else HOURS) + 1, HOURS))
    for h in range(width - 1, -1, -1):
        right_length[:, h] = np.where(positive[:, h], (right_length[:, h+1] if h < width-1 else 0) + 1, 0)
        right_sum[:, h] = np.where(positive[:, h], (right_sum[:, h+1] if h < width-1 else 0) + margin[:, h], 0)
        next_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((next_positive[:, h+1] if h < width-1 else HOURS) + 1, HOURS))
    suffix_count = np.arange(width, 0, -1)
    suffix_positive = positive[:, ::-1].cumsum(1)[:, ::-1]
    suffix_margin = positive_margin[:, ::-1].cumsum(1)[:, ::-1]
    result = {
        "economic_margin_over_week_price_scale": margin / (price.std(1, keepdims=True) + 1.0),
        "economic_positive_streak_elapsed_h": left_length,
        "economic_positive_streak_remaining_h": right_length,
        "economic_positive_streak_total_h": left_length + right_length - positive,
        "economic_positive_streak_margin_sum": left_sum + right_sum - positive_margin,
        "economic_hours_to_next_positive": next_positive,
        "economic_hours_since_previous_positive": previous_positive,
        "economic_positive_fraction_remaining": suffix_positive / suffix_count,
        "economic_positive_margin_remaining_mean": suffix_margin / suffix_count,
        "economic_better_remaining_margin": np.maximum.accumulate(margin[:, ::-1], axis=1)[:, ::-1] - margin,
    }
    indices = np.arange(width)
    for window in (6, 24):
        end = np.minimum(indices + window, width)
        start = np.maximum(indices - window + 1, 0)
        prefix = np.concatenate([np.zeros((n, 1)), margin.cumsum(1)], axis=1)
        result[f"economic_next_{window}h_margin_mean"] = (prefix[:, end] - prefix[:, indices]) / (end - indices)
        result[f"economic_previous_{window}h_margin_mean"] = (prefix[:, indices + 1] - prefix[:, start]) / (indices + 1 - start)
    return result


def water_context(initial, cumulative, remaining, limits, capacities):
    """Local natural-inflow scenarios with zero releases/transfers, not future storage."""
    initial = np.asarray(initial, dtype=np.float64)
    cumulative = np.asarray(cumulative, dtype=np.float64)
    remaining = np.asarray(remaining, dtype=np.float64)
    limits = np.asarray(limits, dtype=np.float64)
    capacities = np.asarray(capacities, dtype=np.float64)
    if cumulative.shape != remaining.shape or limits.shape != cumulative.shape or             cumulative.shape[1:] != (HOURS, len(capacities)) or initial.shape != (len(cumulative), len(capacities)):
        raise ValueError("Invalid water-context dimensions.")
    if (capacities <= 0).any():
        raise ValueError("Reservoir capacities must be positive.")
    capacity = capacities.sum()
    start = initial[:, None, :]
    no_release = start + cumulative
    end_no_release = no_release + remaining
    headroom = np.maximum(capacities - start, 0)
    # A 1% capacity floor bounds the ratio near a full reservoir; apply per reservoir
    # before aggregation so spare space elsewhere does not cancel local pressure.
    pressure = np.log1p((cumulative + remaining) / np.maximum(headroom, 0.01 * capacities))
    future_floor = np.maximum.accumulate(limits[:, ::-1, :], axis=1)[:, ::-1, :]
    return {
        "water_week_inflow_to_headroom_log_max": pressure.max(2),
        "water_week_inflow_to_headroom_log_weighted": (pressure * capacities).sum(2) / capacity,
        "water_no_release_excess_fraction": np.maximum(no_release - capacities, 0).sum(2) / capacity,
        "water_end_no_release_excess_fraction": np.maximum(end_no_release - capacities, 0).sum(2) / capacity,
        "water_no_release_excess_max_fraction": np.maximum((no_release - capacities) / capacities, 0).max(2),
        "water_initial_margin_to_floor_fraction": (start - limits).sum(2) / capacity,
        "water_initial_margin_to_future_floor_fraction": (start - future_floor).sum(2) / capacity,
        "water_no_release_floor_deficit_fraction": np.maximum(limits - no_release, 0).sum(2) / capacity,
        "water_no_release_min_margin_fraction": ((no_release - limits) / capacities).min(2),
    }


def lookup(key, timestamps, columns=None):
    frame = tables[key] if columns is None else tables[key][columns]
    selected = frame.reindex(pd.DatetimeIndex(timestamps))
    if selected.isna().any().any():
        raise ValueError(f"Missing values while looking up {key}")
    return selected.to_numpy(dtype=np.float32)

def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    if len(starts) == 0:
        raise ValueError("Cannot construct features for zero cases.")
    n = len(starts)
    # Use datetime arithmetic rather than assuming pandas timestamps use nanoseconds.
    timestamps = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    price = lookup("prices", timestamps).reshape(n, HOURS)
    initial = lookup("volumes", starts, RESERVOIRS)
    terminal = lookup("water_values", starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflows = lookup("inflows", timestamps).reshape(n, HOURS, -1)
    features = {}

    def add(name, values):
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n,):
            values = np.repeat(values, HOURS)
        elif values.shape == (n, HOURS):
            values = values.reshape(-1)
        if values.shape != (n * HOURS,) or not np.isfinite(values).all():
            raise ValueError(f"Invalid feature {name}: {values.shape}")
        features[name] = values

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", timestamps.hour.to_numpy())
    add("day_of_week", timestamps.dayofweek.to_numpy())
    add("month", timestamps.month.to_numpy())
    add("day_of_year_sin", np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("day_of_year_cos", np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("price", price)
    for stat, values in (("mean", price.mean(1)), ("std", price.std(1)),
                         ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", values)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, values in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(values, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    # Clip at the case boundary, not at the global dataset boundary.
    for offset in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{offset:+d}", price[:, np.clip(np.arange(HOURS) + offset, 0, HOURS - 1)])
    for j, reservoir in enumerate(RESERVOIRS):
        add(f"initial_volume__{reservoir}", initial[:, j])
        add(f"initial_fraction__{reservoir}", initial[:, j] / CAPACITY[reservoir])
        add(f"terminal_value__{reservoir}", terminal[:, j])
        add(f"price_minus_value__{reservoir}", price - terminal[:, j, None])
    for j, name in enumerate(tables["inflows"].columns):
        flow = inflows[:, :, j]
        cumulative = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cumulative_Mm3__{name}", cumulative)
        add(f"inflow_remaining_Mm3__{name}", cumulative[:, -1:] - cumulative)
    for key in ("min_flow", "min_volume"):
        limits = lookup(key, timestamps).reshape(n, HOURS, -1)
        for j, name in enumerate(tables[key].columns):
            add(f"{key}__{name}", limits[:, :, j])
            add(f"{key}_week_max__{name}", limits[:, :, j].max(1))
            if key == "min_volume":
                add(f"initial_margin_to_min__{name}", initial[:, RESERVOIRS.index(name), None] - limits[:, :, j])
    if FEATURE_VERSION == "hydro-context-v2":
        # Rolling summaries stay strictly inside each supplied planning horizon.
        add("price_past_24h_mean", pd.DataFrame(price.T).rolling(24, min_periods=1).mean().to_numpy().T)
        add("price_next_24h_mean", pd.DataFrame(price[:, ::-1].T).rolling(24, min_periods=1).mean().to_numpy().T[:, ::-1])
        add("price_suffix_max", np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        add("price_suffix_min", np.minimum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        for j, reservoir in enumerate(RESERVOIRS):
            curve = topology["model"]["reservoir"][reservoir]["vol_head"]
            curve_x, curve_y = np.asarray(curve["x"]), np.asarray(curve["y"])
            if len(curve_x) != len(curve_y) or not np.all(np.diff(curve_x) > 0):
                raise ValueError(f"Invalid volume-level curve for {reservoir}")
            # np.interp clips outside the supplied curve: a static descriptor only.
            add(f"initial_level_m__{reservoir}", np.interp(initial[:, j], curve_x, curve_y))
    return pd.DataFrame(features, dtype=np.float32)

def generator_features(base, generator):
    # Copy only one generator's matrix at a time; never materialize all 14 together.
    X = base.copy()
    feeders = FEEDERS[generator]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_initial = sum(base[f"initial_volume__{r}"] for r in feeders)
    value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in feeders)
    remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in feeders)
    X["local_initial_fraction"] = total_initial / capacity
    X["local_terminal_value_capacity_weighted"] = value
    X["local_price_minus_terminal_value"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cumulative_inflow_fraction"] = cumulative / capacity
    X["local_remaining_inflow_fraction"] = remaining / capacity
    X["local_available_water_proxy_fraction"] = (total_initial + cumulative + remaining) / capacity
    if FEATURE_VERSION == "hydro-context-v2":
        upstream = UPSTREAM[generator]
        upstream_capacity = sum(CAPACITY[r] for r in upstream)
        upstream_initial = sum(base[f"initial_volume__{r}"] for r in upstream)
        upstream_cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in upstream)
        upstream_remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in upstream)
        upstream_value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in upstream) / upstream_capacity
        X["upstream_initial_fraction"] = upstream_initial / upstream_capacity
        X["upstream_terminal_value_capacity_weighted"] = upstream_value
        X["upstream_price_minus_value"] = base["price"] - upstream_value
        X["upstream_inflow"] = sum(base[f"inflow__{r}"] for r in upstream)
        X["upstream_cumulative_inflow_fraction"] = upstream_cumulative / upstream_capacity
        X["upstream_remaining_inflow_fraction"] = upstream_remaining / upstream_capacity
        X["upstream_available_water_proxy_fraction"] = (upstream_initial + upstream_cumulative + upstream_remaining) / upstream_capacity
        initial_level = sum(base[f"initial_level_m__{r}"] * CAPACITY[r] for r in feeders) / capacity
        X["local_initial_gross_head_proxy_m"] = initial_level - OUTLET_ELEVATION[generator]
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        positive = (margin > 0).astype(np.float32)
        X["local_hours_above_value_remaining"] = positive[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
        X["local_positive_margin_remaining"] = np.maximum(margin, 0)[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
    extra = {}
    if "economics" in DECISION_GROUPS:
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        extra.update(economic_context(margin, base["price"].to_numpy().reshape(-1, HOURS)))
    if "water_balance" in DECISION_GROUPS:
        def cube(prefix):
            return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS) for r in feeders], axis=2)
        initial = cube("initial_volume")[:, 0, :]
        cumulative_cube = cube("inflow_cumulative_Mm3")
        remaining_cube = cube("inflow_remaining_Mm3")
        floor = np.stack([base[f"min_volume__{r}"].to_numpy().reshape(-1, HOURS)
                          if f"min_volume__{r}" in base else np.zeros_like(cumulative_cube[:, :, 0])
                          for r in feeders], axis=2)
        extra.update(water_context(initial, cumulative_cube, remaining_cube, floor,
                                   [CAPACITY[r] for r in feeders]))
    if extra:
        X = pd.concat([X, pd.DataFrame({name: values.reshape(-1).astype(np.float32)
                                       for name, values in extra.items()}, index=X.index)], axis=1)
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features must be finite.")
    return X.astype(np.float32, copy=False)

## 4. Chronological folds and executable data checks

For each development fold, training horizons end no later than the validation start.
Validation horizons also end before the next period starts. Never randomly split the
expanded hourly rows. The two default development folds evaluate 2020 and 2021.

2022 is reserved for evaluating a frozen candidate. It is not used for early stopping
or choosing tree counts. The deleted seasonal baseline previously scored 0.564993 on
2022; that historical result is not directly comparable to the development folds.
The notebook computes a fold-matched seasonal baseline during development evaluation.

In [ ]:
def chronological_split(start, end):
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    if start >= end or end > HOLDOUT_START:
        raise ValueError("Development folds must end by the reserved holdout start.")
    horizon_end = train_cases["starttime"] + pd.Timedelta(hours=HOURS)
    fitting = train_cases.loc[horizon_end <= start].copy()
    validation = train_cases.loc[(train_cases["starttime"] >= start) & (horizon_end <= end)].copy()
    if fitting.empty or validation.empty:
        raise ValueError("Empty chronological fold.")
    if (fitting["starttime"] + pd.Timedelta(hours=HOURS)).max() > validation["starttime"].min():
        raise ValueError("Training and validation horizons overlap.")
    return fitting, validation

def even_sample(cases, maximum):
    indices = np.linspace(0, len(cases) - 1, min(maximum, len(cases)), dtype=int)
    return cases.iloc[indices].copy()

def assemble_predictions(cases, predictions):
    if set(predictions) != set(GENERATORS):
        raise ValueError("Predictions must cover all 14 generators.")
    values = np.empty((len(cases), len(LABELS)), dtype=np.float32)
    for generator in GENERATORS:
        probability = np.asarray(predictions[generator])
        if probability.shape != (len(cases), HOURS):
            raise ValueError(f"Wrong prediction shape for {generator}")
        values[:, TARGET_POSITIONS[generator]] = probability
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Predictions must be finite probabilities in [0, 1].")
    result = pd.DataFrame(values, columns=LABELS)
    result.insert(0, "Run No", cases["Run No"].to_numpy())
    return result

display(pd.DataFrame([{"validation_start": start, "validation_end_exclusive": end,
                       "training_cases": len(chronological_split(start, end)[0]),
                       "validation_cases": len(chronological_split(start, end)[1])}
                      for start, end in FOLDS]))
probe_cases = pd.concat([train_cases.iloc[[0, 1]], test_cases.iloc[[0, -1]]], ignore_index=True)
probe = feature_matrix(probe_cases)
probe_single = feature_matrix(probe_cases.iloc[[1]])
np.testing.assert_allclose(probe.iloc[HOURS:2 * HOURS].to_numpy(), probe_single.to_numpy())
for i, start in enumerate(probe_cases["starttime"]):
    rows = slice(i * HOURS, (i + 1) * HOURS)
    times = pd.date_range(start, periods=HOURS, freq="h")
    np.testing.assert_allclose(probe.iloc[rows]["price"], lookup("prices", times).reshape(-1))
    for reservoir in RESERVOIRS:
        np.testing.assert_allclose(probe.iloc[rows][f"initial_volume__{reservoir}"], tables["volumes"].at[start, reservoir])
        np.testing.assert_allclose(probe.iloc[rows][f"terminal_value__{reservoir}"], tables["water_values"].at[start + pd.Timedelta(hours=HOURS), reservoir])
for g in GENERATORS:
    generator_features(probe, g)
    truth = labels_for(train_cases.iloc[:2], g).reshape(2, HOURS)
    np.testing.assert_array_equal(truth[:, [0, 24, 167]], train_cases.iloc[:2][[TARGET_COLUMNS[g][h] for h in (0, 24, 167)]])
roundtrip = assemble_predictions(train_cases.iloc[:2],
    {g: labels_for(train_cases.iloc[:2], g).reshape(2, HOURS) for g in GENERATORS})
np.testing.assert_array_equal(roundtrip[LABELS], train_cases.iloc[:2][LABELS])
NUMERIC_FEATURE_NAMES = generator_features(probe, GENERATORS[0]).columns.tolist()
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + (["generator_id"] if MODEL_STRATEGY == "shared" else [])
MODEL_KEYS = ["shared"] if MODEL_STRATEGY == "shared" else GENERATORS
FEATURE_SIGNATURE = hashlib.sha256(json.dumps(FEATURE_NAMES).encode()).hexdigest()
if any(c in FEATURE_NAMES for c in ("Run No", "starttime", "total_calculation_time")):
    raise ValueError("Metadata leaked into predictors.")
bytes_per_case = HOURS * len(NUMERIC_FEATURE_NAMES) * np.dtype("float32").itemsize
print(f"Data checks passed: {len(FEATURE_NAMES)} features per generator-hour.")
print(f"One full training feature matrix: approximately {len(train_cases) * bytes_per_case / 2**20:.0f} MiB.")
print("This is not peak memory: copies, CatBoost pools, and model state add overhead.")
del probe, probe_single, roundtrip
if any(re.search(r'[\[\]{}":,]', name) for name in FEATURE_NAMES):
    raise ValueError("A feature name contains unsupported LightGBM JSON characters.")
print("LightGBM check mode audits preprocessing without fitting or importing LightGBM.")


## 5. Training, metrics and interpretation helpers

LightGBM builds asymmetric, leaf-wise gradient-boosted trees. This first configuration
uses depth 7 and at most 63 leaves, learning rate 0.05, minimum 200 rows per leaf,
L2 regularization 5, row/feature subsampling 0.8 and at most 4 CPU threads. These are
starting values, not proven optima. `subsample_freq=1` activates row subsampling.
No class weighting is applied. The number of correlated hourly rows is not the
number of independent cases.

Validation callbacks stop on binary Logloss with patience 100 and retain the best
iteration. Overall model comparisons use micro ROC-AUC, not per-generator average
AUC. A held-out constant-class subset can still be evaluated by Logloss; a constant
training target receives a constant-probability fallback. In holdout/submit modes,
evaluation labels are not accessed during fitting and no early stopping occurs.

Gain importance is normalized to 100% within each generator; the mean of these
percentages is plotted. This differs from CatBoost's importance definition and
must not be numerically compared across libraries as causal evidence.


The neutral-initialization experiment tests a hypothesis suggested by Haukeli's
2020 training/validation ON-fraction shift (0.726639 to 0.244709) and the model's
mean prediction 0.702003 after six retained trees. More trees alone are not a
justified fix: its saved validation loss worsened after the early optimum.


In [ ]:
def safe_auc(actual, predicted):
    from sklearn.metrics import roc_auc_score
    actual = np.asarray(actual).reshape(-1)
    if np.unique(actual).size < 2:
        return None
    return float(roc_auc_score(actual, np.asarray(predicted).reshape(-1)))

def evaluate(cases, predictions):
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    frame = assemble_predictions(cases, predictions)
    actual = cases[LABELS].to_numpy(dtype=np.uint8)
    predicted = frame[LABELS].to_numpy()
    overall = float(roc_auc_score(actual, predicted, average="micro"))
    detail = pd.DataFrame([{"generator": g,
        "auc": safe_auc(labels_for(cases, g), predictions[g]),
        "logloss": float(log_loss(labels_for(cases, g), predictions[g].reshape(-1), labels=[0, 1])),
        "brier": float(brier_score_loss(labels_for(cases, g), predictions[g].reshape(-1))),
        "observed_on_fraction": float(labels_for(cases, g).mean()),
        "mean_prediction": float(predictions[g].mean())} for g in GENERATORS])
    return overall, detail, frame

def seasonal_predictions(fitting, validation):
    groups = fitting[LABELS].groupby(fitting["starttime"].dt.month).mean()
    if not set(validation["starttime"].dt.month).issubset(groups.index):
        raise ValueError("The seasonal reference lacks a validation month in training.")
    values = groups.loc[validation["starttime"].dt.month]
    return {g: values[TARGET_COLUMNS[g]].to_numpy(dtype=np.float32) for g in GENERATORS}

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")

def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def fit_and_predict(fitting, evaluation, parameters, iterations, folder, *, early_stop):
    import lightgbm as lgb
    started = perf_counter()
    folder.mkdir(parents=True, exist_ok=False)
    X_fit_base, X_eval_base = feature_matrix(fitting), feature_matrix(evaluation)
    feature_seconds = perf_counter() - started
    predicted, records, importance, curves = {}, [], [], {}
    for generator in GENERATORS:
        generator_started = perf_counter()
        X_fit = generator_features(X_fit_base, generator)
        X_eval = generator_features(X_eval_base, generator)
        if X_fit.columns.tolist() != FEATURE_NAMES or X_eval.columns.tolist() != FEATURE_NAMES:
            raise ValueError("Feature schema changed between partitions.")
        y = labels_for(fitting, generator)
        limit = int(iterations[generator] if isinstance(iterations, dict) else iterations)
        if limit < 1:
            raise ValueError("Tree counts must be positive.")
        model = None
        if np.unique(y).size == 1:
            constant = float(y[0])
            probability = np.full(len(evaluation) * HOURS, constant, dtype=np.float32)
            tree_count = 1
            write_json(folder / f"{generator}_constant.json", {"probability": constant})
        else:
            constant = None
            model = lgb.LGBMClassifier(**parameters, n_estimators=limit,
                                       random_state=SEED, n_jobs=THREADS, importance_type="gain")
            history = {}
            callbacks = [lgb.log_evaluation(period=100), lgb.record_evaluation(history)]
            options = {"callbacks": callbacks}
            if early_stop:
                # Only the development target is read. Holdout/test labels are untouched.
                options.update(eval_set=[(X_eval, labels_for(evaluation, generator))],
                               eval_names=["validation"], eval_metric="binary_logloss")
                callbacks.append(lgb.early_stopping(EARLY_STOPPING, first_metric_only=True))
            print(f"Training {generator}: {len(y):,} hourly rows, at most {limit} trees", flush=True)
            model.fit(X_fit, y, **options)
            if list(model.classes_) != [0, 1]:
                raise ValueError("Unexpected LightGBM class order.")
            tree_count = int(model.best_iteration_ or model.booster_.current_iteration())
            if not 1 <= tree_count <= limit:
                raise ValueError("Invalid LightGBM retained tree count.")
            probability = model.predict_proba(X_eval, num_iteration=tree_count,
                                               validate_features=True)[:, 1].astype(np.float32)
            model.booster_.save_model(str(folder / f"{generator}.txt"), num_iteration=tree_count)
            curves[generator] = history
            gains = model.booster_.feature_importance(importance_type="gain", iteration=tree_count)
            normalized = 100 * gains / gains.sum() if gains.sum() > 0 else np.zeros_like(gains)
            importance.extend({"generator": generator, "feature": name,
                               "importance": float(value), "gain": float(gain)}
                              for name, value, gain in zip(FEATURE_NAMES, normalized, gains))
        predicted[generator] = probability.reshape(len(evaluation), HOURS)
        records.append({"generator": generator, "trees": tree_count,
                        "constant_probability": constant, "seconds": perf_counter() - generator_started})
        pd.DataFrame(records).to_csv(folder / "training_progress.csv", index=False)
        del X_fit, X_eval, model
        gc.collect()
    pd.DataFrame(importance, columns=["generator", "feature", "importance", "gain"]).to_csv(
        folder / "feature_importance.csv", index=False)
    write_json(folder / "learning_curves.json", curves)
    write_json(folder / "timing.json", {"feature_seconds": feature_seconds,
                                       "total_seconds": perf_counter() - started})
    del X_fit_base, X_eval_base
    gc.collect()
    return predicted, records, importance


## 6. Run the selected mode

`check` stops short of all fitting. `smoke` uses evenly spaced **whole cases**, never
random hourly rows, and cannot produce a final submission or a frozen configuration.
Its score is a plumbing diagnostic, not a full-data performance estimate.

`validate` trains the default development folds and selects the median best tree count across folds, either
for each generator separately or once for the shared model. This is a starting refit rule, not a tuned search.
If many models reach the maximum tree count, inspect convergence before increasing
the budget. Freeze both parameters and tree counts before running `holdout`.

The frozen configuration must also preserve feature schema and random seed. Modifying
feature logic requires incrementing `FEATURE_VERSION` and revalidating. No holdout
score is used to update parameters or iterations automatically.

In [ ]:
RUN_DIRECTORY = None
run_summary = None
importance_rows = []
if RUN_MODE == "check":
    print("Check mode complete. No models trained and no submission written.")
else:
    if RUN_MODE in {"holdout", "submit"}:
        if not isinstance(SELECTED_CONFIG, dict):
            raise ValueError("Paste a completed validate run's configuration into SELECTED_CONFIG first.")
        if SELECTED_CONFIG.get("model_family") != MODEL_FAMILY:
            raise ValueError("Use a completed LightGBM configuration, not a CatBoost configuration.")
        if SELECTED_CONFIG.get("feature_version") != FEATURE_VERSION or SELECTED_CONFIG.get("feature_signature") != FEATURE_SIGNATURE:
            raise ValueError("Frozen configuration and current features do not match.")
        if SELECTED_CONFIG.get("experiment", "baseline_logloss") != EXPERIMENT:
            raise ValueError("Set EXPERIMENT to the frozen configuration's experiment name before holdout or submit.")
        if SELECTED_CONFIG.get("seed") != SEED:
            raise ValueError("Use the same random seed as the frozen configuration.")
        if SELECTED_CONFIG.get("model_strategy", "per_generator") != MODEL_STRATEGY:
            raise ValueError("Frozen configuration uses a different model strategy.")
        if set(SELECTED_CONFIG.get("iterations", {})) != set(MODEL_KEYS):
            raise ValueError(f"Frozen configuration must contain tree counts for {MODEL_KEYS}.")
        if any(not isinstance(v, int) or v < 1 for v in SELECTED_CONFIG["iterations"].values()):
            raise ValueError("Frozen tree counts must be positive integers.")
        parameters = dict(SELECTED_CONFIG["parameters"])
        if parameters.get("device_type") != "cpu":
            raise ValueError("This notebook uses CPU LightGBM training.")
    else:
        parameters = dict(BASE_PARAMETERS)

    if set(parameters) & {"n_estimators", "num_iterations", "num_iteration", "num_boost_round",
                          "iterations", "random_state", "seed", "n_jobs", "num_threads",
                          "num_thread", "early_stopping_round", "early_stopping_rounds",
                          "early_stopping", "importance_type"}:
        raise ValueError("Keep tree budget, seed, threads and stopping outside the frozen parameter dictionary.")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    RUN_DIRECTORY = ROOT / "outputs" / "lightgbm" / f"{stamp}_{RUN_MODE}_{EXPERIMENT}"
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
    run_summary = {"status": "incomplete", "model_family": MODEL_FAMILY, "mode": RUN_MODE, "experiment": EXPERIMENT, "versions": VERSIONS,
        "seed": SEED, "threads": THREADS, "model_strategy": MODEL_STRATEGY, "feature_version": FEATURE_VERSION,
        "feature_signature": FEATURE_SIGNATURE, "features": FEATURE_NAMES,
        "parameters": parameters, "folds": FOLDS,
        "maximum_iterations": MAX_ITERATIONS, "early_stopping_rounds": EARLY_STOPPING,
        "selected_configuration": SELECTED_CONFIG,
        "input_sha256": {k: file_hash(v) for k, v in RAW_FILES.items()}}
    write_json(RUN_DIRECTORY / "run.json", run_summary)

    if RUN_MODE in {"smoke", "validate"}:
        fold_results, iteration_records = [], []
        active_folds = FOLDS[-1:] if RUN_MODE == "smoke" else FOLDS
        for start, end in active_folds:
            fitting, validation = chronological_split(start, end)
            if RUN_MODE == "smoke":
                fitting, validation = even_sample(fitting, 96), even_sample(validation, 48)
            folder = RUN_DIRECTORY / f"fold_{start[:4]}"
            predicted, records, importance = fit_and_predict(
                fitting, validation, parameters, 40 if RUN_MODE == "smoke" else MAX_ITERATIONS,
                folder, early_stop=True)
            auc, detail, prediction_frame = evaluate(validation, predicted)
            baseline_auc, _, _ = evaluate(validation, seasonal_predictions(fitting, validation))
            detail.to_csv(folder / "per_generator_metrics.csv", index=False)
            prediction_frame.to_csv(folder / "validation_predictions.csv", index=False)
            iteration_records.extend(records)
            importance_rows.extend(dict(row, fold=start[:4]) for row in importance)
            fold_results.append({"validation_start": start, "validation_end_exclusive": end,
                                 "training_cases": len(fitting), "validation_cases": len(validation),
                                 "micro_auc": auc, "seasonal_micro_auc": baseline_auc,
                                 "mean_generator_logloss": float(detail["logloss"].mean()),
                                 "mean_generator_brier": float(detail["brier"].mean())})
            print(f"{start[:4]} micro ROC-AUC: {auc:.6f}; same-fold seasonal reference: {baseline_auc:.6f}")
            display(detail)
            del predicted, prediction_frame
        run_summary["development_results"] = fold_results
        display(pd.DataFrame(fold_results))
        if RUN_MODE == "validate":
            selected = {"model_family": MODEL_FAMILY, "experiment": EXPERIMENT, "model_strategy": MODEL_STRATEGY, "feature_version": FEATURE_VERSION, "feature_signature": FEATURE_SIGNATURE,
                        "seed": SEED, "parameters": parameters,
                        "iterations": {g: max(1, int(np.median([r["trees"] for r in iteration_records if r["generator"] == g]))) for g in MODEL_KEYS}}
            write_json(RUN_DIRECTORY / "selected_configuration.json", selected)
            print("Copy this complete dictionary into SELECTED_CONFIG before holdout or submit:")
            print(repr(selected))
            if any(r["trees"] >= MAX_ITERATIONS for r in iteration_records):
                print("Some models reached MAX_ITERATIONS. Inspect their learning curves before freezing the budget.")
    elif RUN_MODE == "holdout":
        fitting = train_cases.loc[train_cases["starttime"] + pd.Timedelta(hours=HOURS) <= HOLDOUT_START]
        holdout = train_cases.loc[train_cases["starttime"] >= HOLDOUT_START]
        if fitting.empty or holdout.empty:
            raise ValueError("Empty holdout partition.")
        predicted, _, importance_rows = fit_and_predict(fitting, holdout, parameters,
            SELECTED_CONFIG["iterations"], RUN_DIRECTORY / "holdout", early_stop=False)
        auc, detail, prediction_frame = evaluate(holdout, predicted)
        prediction_frame.to_csv(RUN_DIRECTORY / "holdout_predictions.csv", index=False)
        detail.to_csv(RUN_DIRECTORY / "holdout_metrics.csv", index=False)
        run_summary["holdout_micro_auc"] = auc
        print(f"Frozen-configuration holdout micro ROC-AUC: {auc:.6f}")
        display(detail)
    elif RUN_MODE == "submit":
        predicted, _, importance_rows = fit_and_predict(train_cases, test_cases, parameters,
            SELECTED_CONFIG["iterations"], RUN_DIRECTORY / "full_training", early_stop=False)
        submission = assemble_predictions(test_cases, predicted)
        if not submission.columns.equals(template.columns) or not submission["Run No"].equals(template["Run No"]):
            raise ValueError("Submission IDs or column order differ from the template.")
        if submission.shape != template.shape:
            raise ValueError("Submission shape differs from the template.")
        destination = RUN_DIRECTORY / "lightgbm_submission.csv"
        submission.to_csv(destination, index=False)
        reloaded = pd.read_csv(destination)
        np.testing.assert_allclose(reloaded[LABELS], submission[LABELS], rtol=1e-6, atol=1e-8)
        np.testing.assert_array_equal(reloaded["Run No"], template["Run No"])
        run_summary["submission"] = str(destination.relative_to(ROOT))
        print(f"Verified submission saved: {destination}")

    run_summary["status"] = "complete"
    run_summary["total_seconds"] = perf_counter() - NOTEBOOK_STARTED
    write_json(RUN_DIRECTORY / "run.json", run_summary)
    (RUN_DIRECTORY / "environment.txt").write_text(
        "\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n", encoding="utf-8")
    print(f"Recorded runtime so far: {run_summary['total_seconds'] / 60:.1f} minutes")
    print("Run artifacts:", RUN_DIRECTORY)

## 7. Inspect interpretation and runtime

Mean normalized split-gain percentages describe predictive association. Correlated
features can redistribute importance; percentages are not causal explanations and
are not directly comparable with CatBoost importance. Keep unsuccessful results
and measure whole-notebook runtime before selecting a final configuration.


In [ ]:
if importance_rows:
    import matplotlib.pyplot as plt
    importance_table = pd.DataFrame(importance_rows)
    importance_table.to_csv(RUN_DIRECTORY / "feature_importance_all.csv", index=False)
    top = importance_table.groupby("feature")["importance"].mean().nlargest(20).sort_values()
    fig, ax = plt.subplots(figsize=(10, 7))
    top.plot.barh(ax=ax)
    ax.set(title="Mean feature importance across trained generator models",
           xlabel="Mean normalized split gain (%)", ylabel="Feature")
    fig.tight_layout()
    fig.savefig(RUN_DIRECTORY / "feature_importance.png", dpi=150)
    plt.show()
if RUN_DIRECTORY is not None:
    run_summary["total_seconds"] = perf_counter() - NOTEBOOK_STARTED
    write_json(RUN_DIRECTORY / "run.json", run_summary)
    print(f"Complete notebook runtime: {run_summary['total_seconds'] / 60:.1f} minutes")
else:
    print("No training results yet. Change RUN_MODE yourself when ready.")

## 8. Compare development runs across model families

Read completed 2020/2021 development results from CatBoost and LightGBM only.
Match input hashes, fold definitions, feature signature and random seed before
attributing differences to the learner. Model-specific parameters and capacities
are deliberately different and require their own tuning. Older 193-feature runs
are useful references but do not isolate the learner on the 216-feature pipeline.
No holdout or Kaggle score is used by this comparison to select models.


In [ ]:
current_hashes = run_summary["input_sha256"] if run_summary else {key: file_hash(path) for key, path in RAW_FILES.items()}
comparison_rows = []
metadata_paths = sorted((ROOT / "outputs/catboost").glob("*/run.json")) + sorted(
    (ROOT / "outputs/lightgbm").glob("*/run.json"))
for metadata_path in metadata_paths:
    previous = json.loads(metadata_path.read_text(encoding="utf-8"))
    if previous.get("mode") != "validate" or previous.get("status") != "complete":
        continue
    comparable = (previous.get("feature_signature") == FEATURE_SIGNATURE
                  and previous.get("folds") == [list(fold) for fold in FOLDS]
                  and previous.get("seed") == SEED
                  and previous.get("input_sha256") == current_hashes)
    for result in previous.get("development_results", []):
        comparison_rows.append({"run": metadata_path.parent.name,
            "model_family": previous.get("model_family", metadata_path.parent.parent.name),
            "experiment": previous.get("experiment", "baseline_auc"),
            "same_inputs_features_folds_seed": comparable,
            "validation_year": result["validation_start"][:4],
            "micro_auc": result["micro_auc"],
            "mean_generator_logloss": result.get("mean_generator_logloss"),
            "mean_generator_brier": result.get("mean_generator_brier"),
            "run_minutes": previous.get("total_seconds", 0) / 60})
if comparison_rows:
    comparison = pd.DataFrame(comparison_rows)
    display(comparison)
    print("Input hashes are included in the comparison flag. Improvement on one year is not conclusive.")
    if RUN_DIRECTORY is not None:
        comparison.to_csv(RUN_DIRECTORY / "development_comparison.csv", index=False)
